In [ ]:
from DEigerClient3 import DEigerClient

url = "http://169.254.254.1/monitor/api/1.8.0/images/monitor"
DCU_IP = '169.254.254.1'
ec = DEigerClient(DCU_IP)

In [ ]:
import pyvisa
from A7_pulser_v2 import A7_pulser
rm = pyvisa.ResourceManager()
resourceList = rm.list_resources()
index = 5
pulser = A7_pulser(resourceList[index])
channelC = {"source": 'TRIG', "divider": 0,
                "delay": 0, "width": 4}  # for inhibiting the SRS
pulser.channel[2].set_source(channelC['source'])
pulser.channel[2].set_divider(channelC['divider'])
pulser.channel[2].set_delay(channelC['delay'])
pulser.channel[2].set_width(channelC['width'])
pulser.channel[2].set_source('TRIG')


In [ ]:
pulser.channel[2].set_source('TRIG')


In [ ]:
filename = 'data_1pulse'
ec.sendDetectorCommand('disarm')
ec.setDetectorConfig('trigger_mode', 'ints')
ec.setDetectorConfig('incident_energy', 40000)  # Ev
ec.setDetectorConfig('ntrigger', 1)  # ntrigger must be set to 1 for extg mode
ec.setDetectorConfig('nimages', 2)  # nimage must be even for extg mode
ec.setFileWriterConfig('nimages_per_file', 2)
ec.setDetectorConfig('nexpi', 1)  # expi, number of total triggers
ec.setDetectorConfig('trigger_mode', 'extg')  # external trigger gatted mode
ec.setDetectorConfig('counting_mode', 'normal')
ec.setDetectorConfig('extg_mode', 'double')  # this is to set pump probe
ec.setDetectorConfig('countrate_correction_applied', False)
ec.setMonitorConfig('mode', 'enabled')
ec.sendFileWriterCommand('clear')
ec.setMonitorConfig('buffer_size', 100)
    # string 'False' to boolean False
ec.setDetectorConfig('count_time', 1)
ec.setDetectorConfig('frame_time', 1)
ec.setFileWriterConfig('name_pattern', 'data_1pulse')
ec.setFileWriterConfig('compression_enabled', True)
ec.setFileWriterConfig('mode', 'enabled')
ec.sendDetectorCommand('disarm')

In [ ]:
import io
import h5py
import requests
import numpy as np
import matplotlib.pyplot as plt
import hdf5plugin
import time

In [ ]:
a = ec.fileWriterFiles()
print(a)

In [ ]:
# ec.sendDetectorCommand('initialize')

In [ ]:
ec.sendDetectorCommand('arm')

In [ ]:
pulser.channel[2].set_source('OFF')

In [ ]:
url = 'http://169.254.254.1/data/'+filename+"_data_000001.h5"
hf = h5py.File(io.BytesIO(requests.get(url).content), "r+")
print(hf['entry'])

In [ ]:
ec.fileWriterFiles()

In [ ]:
ec.sendDetectorCommand('disarm')

In [ ]:
#ec.sendDetectorCommand('initialize')
loop_size = 2
list_img1 = []
list_img2 = []
# a = ec.fileWriterFiles()
# print(a)
# ec.sendFileWriterCommand('clear')
for i in range(loop_size):
    
    ec.sendDetectorCommand('arm')
    time.sleep(1) 
    
    pulser.channel[2].set_source('OFF')
    time.sleep(1) 

    ec.sendDetectorCommand('disarm')
    time.sleep(1) 
    
    pulser.channel[2].set_source('TRIG')
    time.sleep(1) 
    
    url = 'http://169.254.254.1/data/'+filename+"_data_000001.h5"
    hf = h5py.File(io.BytesIO(requests.get(url).content), "r+")
    print(hf['entry'])
    np.array(hf['entry']['data']['data'])
    data = hf['entry']['data']['data'][()]
    list_img1.append(np.array(data[0]))
    list_img2.append(np.array(data[1]))
    time.sleep(1) 
    
    


In [ ]:
list_img1 

In [ ]:
np.max(list_img2[0])

In [ ]:
plt.imshow(list_img2[1], vmin=0, vmax=1)